In [1]:
# 02_baselines.ipynb
# =============================================================================
# RQ1 non-graph baselines. Two tiers on the strict split (mask A primary,
# mask B reference):
#   (i)  basic-information FLOOR: logistic regression and a small MLP on the 3
#        basic company attributes only (no lawsuit, no graph).
#   (ii) INTRA tabular competitors: logistic regression, MLP, LightGBM and
#        XGBoost on the full 23-d node features (3 basic + 20 lawsuit-aggregate,
#        still no graph). These are the strong non-graph reference the graph
#        model must beat, answering the standard "did you try gradient boosting?"
# Stochastic models are averaged over 10 seeds (mean +/- std); logistic
# regression is deterministic. Metrics: AUC, F1, KS. Figure is grayscale,
# dpi 600, png + pdf, no caption.
# =============================================================================
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[0]
PROCESSED = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "savefig.bbox": "tight",
                     "font.size": 10})
sns.set_theme(style="whitegrid", palette="Greys_r")
SEEDS = list(range(10))


def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)


def ks_stat(y, s):
    fpr, tpr, _ = roc_curve(y, s)
    return float(np.max(tpr - fpr))


def metrics(y, s, thr=0.5):
    return {"AUC": roc_auc_score(y, s), "F1": f1_score(y, (s >= thr).astype(int),
            zero_division=0), "KS": ks_stat(y, s)}


# --- Load features -----------------------------------------------------------
data = torch.load(PROCESSED / "smesd_hetero.pt", weights_only=False)
comp = data["company"]
X_all = comp.x.numpy().astype(np.float64)
y_all = comp.y.numpy()
X_all[:, :2] = np.log1p(np.clip(X_all[:, :2], 0, None))     # log capitals
FEATSETS = {"basic3": X_all[:, :3], "intra23": X_all}
masks = {
    "A": {"train": comp.train_mask.numpy(), "val": comp.val_mask.numpy(),
          "test": comp.test_mask.numpy()},
    "B": {"train": comp.train_mask_repro.numpy(), "val": comp.val_mask_repro.numpy(),
          "test": comp.test_mask_repro.numpy()},
}


def scaled(X, tr):
    sc = StandardScaler().fit(X[tr])
    return sc.transform(X)


def make_model(name, seed):
    if name == "LogReg":
        return LogisticRegression(max_iter=1000, class_weight="balanced", random_state=seed)
    if name == "MLP":
        return MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500,
                             early_stopping=True, random_state=seed)
    if name == "LightGBM":
        return LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                              subsample=0.8, colsample_bytree=0.8, class_weight="balanced",
                              random_state=seed, verbose=-1)
    if name == "XGBoost":
        return XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                             subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
                             random_state=seed, verbosity=0)


# floor models use basic3; competitors use intra23
PLAN = [("basic3", "LogReg", "Floor"), ("basic3", "MLP", "Floor"),
        ("intra23", "LogReg", "Intra"), ("intra23", "MLP", "Intra"),
        ("intra23", "LightGBM", "Intra"), ("intra23", "XGBoost", "Intra")]
DETERMINISTIC = {"LogReg"}

rows = []
for mask_tag, m in masks.items():
    tr, te = m["train"], m["test"]
    for fs, name, tier in PLAN:
        X = scaled(FEATSETS[fs], tr)
        seeds = [0] if name in DETERMINISTIC else SEEDS
        aucs, f1s, kss = [], [], []
        for sd in seeds:
            mdl = make_model(name, sd)
            mdl.fit(X[tr], y_all[tr])
            p = mdl.predict_proba(X[te])[:, 1]
            met = metrics(y_all[te], p)
            aucs.append(met["AUC"]); f1s.append(met["F1"]); kss.append(met["KS"])
        rows.append({"mask": mask_tag, "tier": tier, "features": fs, "model": name,
                     "AUC_mean": np.mean(aucs), "AUC_std": np.std(aucs),
                     "F1_mean": np.mean(f1s), "KS_mean": np.mean(kss),
                     "n_seeds": len(seeds)})

res = pd.DataFrame(rows).round(4)
res.to_csv(TAB / "02_baselines.csv", index=False)
print("=== Baselines (test split) ===")
print(res.to_string(index=False))

A_test = res[res["mask"] == "A"].copy()
print("\n=== Mask A (strict), test AUC ===")
for _, r in A_test.iterrows():
    print(f"  {r.tier:5s} {r.model:9s} ({r.features:7s})  "
          f"AUC {r.AUC_mean:.3f}+/-{r.AUC_std:.3f}  F1 {r.F1_mean:.3f}  KS {r.KS_mean:.3f}")
floor_best = A_test[A_test.tier == "Floor"]["AUC_mean"].max()
intra_best = A_test[A_test.tier == "Intra"]["AUC_mean"].max()
print(f"\nbest floor AUC = {floor_best:.3f} ; best intra-tabular AUC = {intra_best:.3f} "
      f"(the graph model in 03 is measured against these)")

# --- Figure: AUC by model (mask A, grayscale) --------------------------------
fig, ax = plt.subplots(figsize=(6.5, 3.4))
order = A_test.sort_values("AUC_mean")
labels = [f"{r.model}\n({r.features})" for _, r in order.iterrows()]
colors = ["#999999" if t == "Floor" else "#333333" for t in order["tier"]]
ax.barh(np.arange(len(order)), order["AUC_mean"], xerr=order["AUC_std"],
        color=colors, edgecolor="black", linewidth=0.5,
        error_kw={"elinewidth": 0.8, "capsize": 2})
ax.set_yticks(np.arange(len(order))); ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("Test AUC (mask A)")
ax.set_xlim(0.5, 0.8)
handles = [plt.Rectangle((0, 0), 1, 1, color="#999999", ec="black"),
           plt.Rectangle((0, 0), 1, 1, color="#333333", ec="black")]
ax.legend(handles, ["Basic-info floor", "Intra tabular (23-d)"], frameon=True, fontsize=8,
          loc="lower right")
savefig(fig, "fig_baselines")

print("\n02_baselines complete: floor + intra-tabular competitors (incl. LightGBM, "
      "XGBoost) saved; 1 figure, 1 table.")


=== Baselines (test split) ===
mask  tier features    model  AUC_mean  AUC_std  F1_mean  KS_mean  n_seeds
   A Floor   basic3   LogReg    0.6235   0.0000   0.7157   0.2388        1
   A Floor   basic3      MLP    0.6564   0.0508   0.7851   0.2789       10
   A Intra  intra23   LogReg    0.6216   0.0000   0.6564   0.2229        1
   A Intra  intra23      MLP    0.6813   0.0087   0.7345   0.3059       10
   A Intra  intra23 LightGBM    0.6747   0.0037   0.6518   0.2745       10
   A Intra  intra23  XGBoost    0.6807   0.0043   0.6837   0.2951       10
   B Floor   basic3   LogReg    0.6207   0.0000   0.7157   0.2439        1
   B Floor   basic3      MLP    0.6339   0.0564   0.7822   0.2546       10
   B Intra  intra23   LogReg    0.6172   0.0000   0.6633   0.2172        1
   B Intra  intra23      MLP    0.6857   0.0113   0.7357   0.3102       10
   B Intra  intra23 LightGBM    0.6794   0.0028   0.6610   0.2701       10
   B Intra  intra23  XGBoost    0.6790   0.0047   0.6810   0.2885    


02_baselines complete: floor + intra-tabular competitors (incl. LightGBM, XGBoost) saved; 1 figure, 1 table.
